# 01 · First model: predicting Premier League scores with a Poisson model

**Goal of this notebook:** download real match data, check that goals really behave like a Poisson distribution, fit the model, predict Big Six matches, and test honestly against the bookmakers.



## 0 · Setup (run every time you open the notebook)

In [1]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
GITHUB_USER = "Sethchiji"
REPO_NAME = "epl-score-predictor"

if IN_COLAB:
    repo_dir = f"/content/{REPO_NAME}"
    if not os.path.exists(repo_dir):
        !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git {repo_dir}
    else:
        !git -C {repo_dir} pull -q
    %cd {repo_dir}
    !pip install -q -r requirements.txt
else:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(repo_dir)

sys.path.insert(0, repo_dir)
print("Working in:", repo_dir)

/content/epl-score-predictor
Working in: /content/epl-score-predictor


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import poisson

from src.data import load_recent, BIG_SIX, season_code, current_season_start
from src.model import fit_poisson
from src.evaluate import backtest, compare

pd.set_option("display.precision", 3)

## 1 · Load the data
We pull the current season plus the previous three from football-data.co.uk. Each row is one match: date, teams, full-time goals (`FTHG`/`FTAG`), result (`FTR` = H/D/A), and bookmaker odds.

In [3]:
matches = load_recent(n_seasons=4)
print(matches.shape)
print(matches["season"].value_counts().sort_index())
matches.tail()

(1240, 16)
season
2324    380
2425    380
2526    380
2627    100
Name: count, dtype: int64


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,PSH,PSD,PSA,B365H,B365D,B365A,AvgH,AvgD,AvgA,season
1235,2026-11-08,Ipswich,Bournemouth,1,2,A,NaN,NaN,NaN,3.8,3.75,1.91,3.72,3.69,1.90,2627
1236,2026-11-08,Sunderland,Chelsea,1,0,H,NaN,NaN,NaN,3.9,3.60,1.91,3.68,3.60,1.93,2627
1237,2026-11-08,Brighton,Brentford,2,0,H,NaN,NaN,NaN,1.9,3.80,3.70,1.90,3.82,3.59,2627
1238,2026-11-08,Crystal Palace,Liverpool,1,3,A,NaN,NaN,NaN,5.5,4.33,1.57,5.15,4.27,1.57,2627
1239,2026-11-08,Man United,Aston Villa,4,4,D,NaN,NaN,NaN,2.1,3.60,3.30,2.13,3.68,3.07,2627


## 2 · Explore: do goals look Poisson?
The whole model rests on this assumption, so we check it before trusting it. If goals are Poisson, the observed share of matches with 0, 1, 2… goals should match the Poisson curve with the same average.

In [4]:
print(f"Average home goals: {matches.FTHG.mean():.2f}")
print(f"Average away goals: {matches.FTAG.mean():.2f}")
print("Result share:", matches.FTR.value_counts(normalize=True).round(3).to_dict())

goals = np.arange(0, 8)
check = pd.DataFrame({
    "goals": np.tile(goals, 2),
    "side": ["Home"] * len(goals) + ["Away"] * len(goals),
    "observed": np.concatenate([
        matches.FTHG.value_counts(normalize=True).reindex(goals, fill_value=0).values,
        matches.FTAG.value_counts(normalize=True).reindex(goals, fill_value=0).values]),
    "poisson": np.concatenate([poisson.pmf(goals, matches.FTHG.mean()),
                               poisson.pmf(goals, matches.FTAG.mean())]),
})
fig = px.bar(check, x="goals", y="observed", color="side", barmode="group",
             title="Goals per team per match: observed (bars) vs Poisson (lines)")
for side, dash in [("Home", "solid"), ("Away", "dash")]:
    s = check[check.side == side]
    fig.add_scatter(x=s.goals, y=s.poisson, mode="lines+markers", name=f"Poisson ({side})", line_dash=dash)
fig.show()

Average home goals: 1.61
Average away goals: 1.36
Result share: {'H': 0.431, 'A': 0.323, 'D': 0.247}


##### How well does the Poisson curve match? Where is it off? (Hint: look at 0 goals and draws. That gap is exactly what the Dixon-Coles adjustment fixes later.)

## 3 · Fit the model
Each team gets an **attack** and a **defense** strength, plus one **home advantage** for the whole league. Recent matches count more: with the default setting, a match from one year ago counts about half as much as yesterday's.

In [5]:
model = fit_poisson(matches)
print(f"Trained on matches up to {model.as_of:%d %b %Y}")
print(f"Home advantage: {model.home_advantage:.2f}x goals")
ratings = model.team_ratings()
ratings

Trained on matches up to 08 Nov 2026
Home advantage: 1.20x goals


,attack,defense,net_rating
Arsenal,1.344,0.535,2.513
Man City,1.491,0.733,2.034
Liverpool,1.350,0.837,1.613
Brighton,1.264,0.835,1.514
Chelsea,1.255,0.955,1.314
Brentford,1.159,0.928,1.249
Bournemouth,1.144,0.923,1.240
Aston Villa,1.180,0.955,1.235
Newcastle,1.180,0.997,1.184
Man United,1.169,0.995,1.175


In [6]:
fig = px.scatter(ratings.reset_index(names="team"), x="attack", y="defense", text="team",
                 title="Team strength (top-right of the chart = weaker defense, so the y-axis is flipped)",
                 labels={"attack": "Attack (higher = better)", "defense": "Defense (lower = better)"})
fig.update_yaxes(autorange="reversed"); fig.update_traces(textposition="top center")
fig.show()

In [7]:
cells = {"0-0": (0, 0), "1-0": (1, 0), "0-1": (0, 1), "1-1": (1, 1)}
pred = {c: 0.0 for c in cells}
pred_draw = 0.0
for h, a in zip(matches.HomeTeam, matches.AwayTeam):
    m = model.score_matrix(h, a).values
    for c, (i, j) in cells.items():
        pred[c] += m[i, j]
    pred_draw += np.trace(m)

n = len(matches)
actual = (matches.FTHG.astype(str) + "-" + matches.FTAG.astype(str)).value_counts()
check = pd.DataFrame({
    "model predicts": {c: pred[c] / n for c in cells},
    "actually happened": {c: actual.get(c, 0) / n for c in cells},
})
check.loc["all draws"] = [pred_draw / n, (matches.FTR == "D").mean()]
check["gap"] = check["actually happened"] - check["model predicts"]
check.style.format("{:.1%}")

,model predicts,actually happened,gap
0-0,5.9%,4.8%,-1.0%
1-0,8.8%,6.5%,-2.3%
0-1,7.5%,7.2%,-0.3%
1-1,10.8%,11.4%,0.6%
all draws,23.2%,24.7%,1.5%


In [8]:
draws = {"0-0": 0, "1-1": 1, "2-2": 2, "3-3+": 3}
rows = {}
for label, k in draws.items():
    p = 0.0
    for h, a in zip(matches.HomeTeam, matches.AwayTeam):
        m = model.score_matrix(h, a).values
        p += m[k, k] if k < 3 else sum(m[i, i] for i in range(3, m.shape[0]))
    actual_pct = ((matches.FTHG == matches.FTAG) & (matches.FTHG >= k) &
                  ((matches.FTHG == k) if k < 3 else True)).mean()
    rows[label] = {"model predicts": p / n, "actually happened": actual_pct}
d = pd.DataFrame(rows).T
d["gap"] = d["actually happened"] - d["model predicts"]
d.style.format("{:.1%}")

,model predicts,actually happened,gap
0-0,5.9%,4.8%,-1.0%
1-1,10.8%,11.4%,0.6%
2-2,5.2%,7.5%,2.3%
3-3+,1.3%,1.0%,-0.4%


## 4 · Predict Big Six matches
We trained on **every** Premier League match (that's what tells us how strong the Big Six really are), and now focus the output on Big Six head-to-heads.

In [9]:
from itertools import permutations
big_six = [t for t in BIG_SIX if t in model.teams]
fixtures = pd.DataFrame(list(permutations(big_six, 2)), columns=["HomeTeam", "AwayTeam"])
preds = model.predict_fixtures(fixtures)
preds[["home_team", "away_team", "exp_home_goals", "exp_away_goals",
       "p_home", "p_draw", "p_away", "most_likely_score"]]

,home_team,away_team,exp_home_goals,exp_away_goals,p_home,p_draw,p_away,most_likely_score
0,Arsenal,Chelsea,1.943,0.844,0.632,0.212,0.156,1-0
1,Arsenal,Liverpool,1.703,0.908,0.561,0.239,0.199,1-0
2,Arsenal,Man City,1.491,1.002,0.485,0.261,0.254,1-1
3,Arsenal,Man United,2.025,0.786,0.663,0.201,0.136,2-0
4,Arsenal,Tottenham,2.058,0.685,0.695,0.193,0.113,2-0
5,Chelsea,Arsenal,1.016,1.614,0.238,0.249,0.513,1-1
6,Chelsea,Liverpool,1.589,1.620,0.376,0.234,0.390,1-1
7,Chelsea,Man City,1.392,1.789,0.300,0.230,0.469,1-1
8,Chelsea,Man United,1.889,1.403,0.490,0.224,0.286,1-1
9,Chelsea,Tottenham,1.921,1.223,0.536,0.222,0.242,1-1


In [10]:
# Scoreline probabilities for one match. change the teams if need to
home, away = "Arsenal", "Man City"
mat = model.score_matrix(home, away, max_goals=5)
px.imshow(mat, text_auto=".1%", color_continuous_scale="Greens",
          labels={"x": f"{away} goals", "y": f"{home} goals", "color": "Probability"},
          title=f"{home} vs {away}: probability of each scoreline").show()
model.predict_match(home, away)

{'home_team': 'Arsenal',
 'away_team': 'Man City',
 'exp_home_goals': 1.4912912233622835,
 'exp_away_goals': 1.0024463573017603,
 'p_home': 0.4850879260759008,
 'p_draw': 0.2606601477702291,
 'p_away': 0.2542513942300102,
 'most_likely_score': '1-1',
 'p_most_likely_score': 0.12348298900148981}

## 5 · Honest evaluation: walk-forward backtest vs the bookmakers
We replay the **last completed season** week by week. Before each week, the model is refit using only matches that had already happened, so it never sees the future.

How to read the results:
- **Log loss** and **Brier score**: lower is better. These judge the *probabilities*, which is what matters.
- **Accuracy**: how often the most likely outcome happened. It sounds intuitive but is a weak metric, because draws are rarely the top pick.
- **Pinnacle odds** are the sharpest bookmaker in the dataset. Getting close to them is a strong result; beating them consistently is very rare.

In [11]:
last_full = season_code(current_season_start() - 1)
print("Backtesting season", last_full, "... (takes ~10-30 seconds)")
bt = backtest(matches, test_season=last_full)
results = compare(bt, train_matches=matches[matches["season"] < last_full])
results

Backtesting season 2526 ... (takes ~10-30 seconds)


,log_loss,brier,accuracy,n_matches,exact_score_hit_rate
Poisson model,1.029,0.617,0.468,378.0,0.119
Baseline (league avg rates),1.087,0.657,0.423,378.0,NaN
Pinnacle odds,0.988,0.591,0.524,208.0,NaN
Bet365 odds,1.018,0.611,0.489,378.0,NaN
Market average odds,1.015,0.610,0.495,378.0,NaN


In [12]:
# Calibration check: when the model says "60% home win", does the home team win about 60% of the time?
bt["bucket"] = pd.cut(bt["p_home"], bins=np.arange(0, 1.01, 0.1))
calib = bt.groupby("bucket", observed=True).agg(predicted=("p_home", "mean"),
                                                 actual=("FTR", lambda s: (s == "H").mean()),
                                                 n=("FTR", "size")).reset_index(drop=True)
fig = px.scatter(calib, x="predicted", y="actual", size="n", title="Calibration: home-win probability")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line_dash="dash")
fig.show()
calib

,predicted,actual,n
0,0.165,0.143,21
1,0.254,0.236,55
2,0.357,0.394,94
3,0.449,0.376,101
4,0.548,0.579,57
5,0.642,0.700,40
6,0.763,0.778,9
7,0.825,1.000,1


In [13]:
# 1) Log loss for each individual match
P = bt[["p_home", "p_draw", "p_away"]].to_numpy()
idx = bt["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
bt["match_log_loss"] = -np.log(np.clip(P[np.arange(len(P)), idx], 1e-12, 1))

# 2) Does it get better as the season goes on?
bt["month"] = bt["Date"].dt.to_period("M")
print("By month:\n", bt.groupby("month")["match_log_loss"].agg(["mean", "size"]).round(3), "\n")

# 3) Are teams with no prior-season data the problem?
prior = matches[matches["season"] < last_full]
known = set(prior.HomeTeam) | set(prior.AwayTeam)
bt["new_team_involved"] = ~bt.home_team.isin(known) | ~bt.away_team.isin(known)
print("New team involved:\n", bt.groupby("new_team_involved")["match_log_loss"].agg(["mean", "size"]).round(3), "\n")
print("Teams with no prior data:", sorted((set(bt.home_team) | set(bt.away_team)) - known), "\n")

# 4) The 10 worst predictions
print(bt.nlargest(10, "match_log_loss")[["Date", "home_team", "away_team", "actual_score",
                                         "p_home", "p_draw", "p_away", "match_log_loss"]].to_string())

# 5) Fair Pinnacle comparison: score everyone on the same 208 matches
ok = bt[["PSH", "PSD", "PSA"]].notna().all(axis=1)
compare(bt[ok], train_matches=prior)


By month:
           mean  size
month               
2025-08  1.043    28
2025-09  1.012    30
2025-10  0.945    30
2025-11  0.909    40
2025-12  0.982    56
2026-01  1.156    49
2026-02  1.063    42
2026-03  1.127    32
2026-04  0.996    30
2026-05  1.036    41 

New team involved:
                     mean  size
new_team_involved             
False              1.013   306
True               1.096    72 

Teams with no prior data: ['Leeds', 'Sunderland'] 

          Date  home_team      away_team actual_score  p_home  p_draw  p_away  match_log_loss
308 2026-04-11    Arsenal    Bournemouth          1-2   0.676   0.193   0.131           2.034
226 2026-01-25    Arsenal     Man United          2-3   0.668   0.201   0.132           2.028
214 2026-01-17  Liverpool        Burnley          1-1   0.755   0.148   0.097           1.911
76  2025-10-19  Liverpool     Man United          1-2   0.665   0.186   0.148           1.907
111 2025-11-22  Liverpool  Nott'm Forest          0-3   0.661   0.1

,log_loss,brier,accuracy,n_matches,exact_score_hit_rate
Poisson model,0.995,0.592,0.510,208.0,0.111
Baseline (league avg rates),1.076,0.650,0.447,208.0,NaN
Pinnacle odds,0.988,0.591,0.524,208.0,NaN
Bet365 odds,0.990,0.592,0.519,208.0,NaN
Market average odds,0.987,0.591,0.524,208.0,NaN


## 6
1. **Tune the time decay.** Re-run the backtest with `xi=0.001` (slower decay) and `xi=0.004` (faster). Which gives the lowest log loss? Example: `backtest(matches, last_full, xi=0.004)`.
2. **Big Six only.** Filter `bt` to matches where both teams are in `BIG_SIX` and run `compare()` on it. Is the model better or worse on these games?


In [14]:
# Section 7: tune shrinkage (how strongly ratings are pulled toward average)
rows = []
for s in [0.5, 1, 2, 5, 10, 20, 40]:
    bt_s = backtest(matches, test_season=last_full, shrinkage=s)
    P = bt_s[["p_home", "p_draw", "p_away"]].to_numpy()
    i = bt_s["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
    worst = (-np.log(P[np.arange(len(P)), i])).max()
    m = compare(bt_s, train_matches=prior).loc["Poisson model"]
    rows.append({"shrinkage": s, "log_loss": m.log_loss, "brier": m.brier,
                 "accuracy": m.accuracy, "worst_match_loss": worst})
tuning = pd.DataFrame(rows)
best = tuning.loc[tuning.log_loss.idxmin(), "shrinkage"]
print("Best shrinkage:", best, "| Bet365 log loss on the same matches:",
      compare(bt, train_matches=prior).loc["Bet365 odds", "log_loss"])
tuning.round(4)

Best shrinkage: 5.0 | Bet365 log loss on the same matches: 1.0183


,shrinkage,log_loss,brier,accuracy,worst_match_loss
0,0.5,1.036,0.620,0.471,3.415
1,1.0,1.033,0.619,0.468,2.846
2,2.0,1.031,0.618,0.471,2.318
3,5.0,1.029,0.617,0.468,2.034
4,10.0,1.029,0.617,0.471,1.954
5,20.0,1.031,0.618,0.479,1.834
6,40.0,1.036,0.621,0.479,1.678


Upcoming fixtures

In [17]:
fx = pd.read_csv("https://www.football-data.co.uk/fixtures.csv", encoding="utf-8-sig")
print("Leagues in this file:", sorted(fx["Div"].unique()))
epl = fx[fx["Div"] == "E0"][["Date", "Time", "HomeTeam", "AwayTeam", "B365H", "B365D", "B365A"]]
print(len(epl), "Premier League fixtures")
epl

Leagues in this file: ['E2', 'E3', 'EC', 'SC1', 'SC2', 'SC3', 'SP2']
0 Premier League fixtures


,Date,Time,HomeTeam,AwayTeam,B365H,B365D,B365A


### ADDING DIFFERENT OPTIONS FOR TEAM TOTAL GOALS

In [16]:
import requests
data = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", timeout=30).json()
players = pd.DataFrame(data["elements"])
teams = {t["id"]: t["name"] for t in data["teams"]}
players["team_name"] = players["team"].map(teams)
cols = ["web_name", "team_name", "status", "chance_of_playing_next_round", "news", "form", "expected_goals", "minutes"]
print(players.shape)
players[players["status"] != "a"][cols].head(15)   # status "a" = available, so these are flagged players

(672, 110)


,web_name,team_name,status,chance_of_playing_next_round,news,form,expected_goals,minutes
11,Saka,Arsenal,i,0.0,Thigh injury - Unknown return date,0.0,3.63,596
17,Martinelli,Arsenal,u,0.0,Has joined Al Hilal permanently,0.0,0.00,0
18,Zubimendi,Arsenal,d,75.0,Hamstring injury - 75% chance of playing,0.7,0.21,100
19,Dowman,Arsenal,s,0.0,Suspended until 29 Nov,0.3,0.00,19
20,Nwaneri,Arsenal,u,0.0,Has joined Borussia Dortmund on loan for the r...,0.0,0.00,0
21,Fábio Vieira,Arsenal,u,0.0,Has joined Hamburger SV permanently,0.0,0.00,0
22,Nelson,Arsenal,u,0.0,has departed the club as a free agent.,0.0,0.00,0
25,G.Jesus,Arsenal,u,0.0,Has joined FC Barcelona permanently,0.0,0.00,0
30,Digne,Aston Villa,u,0.0,Has joined Paris Saint-Germain permanently,0.0,0.00,0
31,Cash,Aston Villa,d,75.0,Concussion - 75% chance of playing,1.3,0.21,656


#### Measuring Players Absesnce impact

In [18]:
players["xg"] = players["expected_goals"].astype(float)
team_xg = players.groupby("team_name")["xg"].sum()
players["xg_share"] = players["xg"] / players["team_name"].map(team_xg)

# chance of missing the next match: injured/suspended = 100%, doubtful = 100 - chance_of_playing
missing = players[players["status"].isin(["i", "d", "s"]) & (players["minutes"] > 0)].copy()
missing["p_missing"] = (100 - missing["chance_of_playing_next_round"].fillna(0)) / 100

missing["xg_share_lost"] = missing["xg_share"] * missing["p_missing"]
print("Biggest absences by attacking impact:")
display(missing.sort_values("xg_share_lost", ascending=False)
        [["web_name", "team_name", "status", "news", "xg", "xg_share", "p_missing", "xg_share_lost"]].head(15))

print("\nTotal share of team xG currently missing:")
missing.groupby("team_name")["xg_share_lost"].sum().sort_values(ascending=False).round(3)

Biggest absences by attacking impact:


,web_name,team_name,status,news,xg,xg_share,p_missing,xg_share_lost
116,Thiago,Brentford,s,Suspended until 5 Dec,4.12,0.276,1.00,0.276
11,Saka,Arsenal,i,Thigh injury - Unknown return date,3.63,0.225,1.00,0.225
185,João Pedro,Chelsea,i,Thigh injury - Unknown return date,3.05,0.216,1.00,0.216
258,Mateta,Crystal Palace,s,Suspended until 2 Dec,1.96,0.142,1.00,0.142
665,Brobbey,Sunderland,d,Muscular injury - 75% chance of playing,4.21,0.268,0.25,0.067
411,Stach,Leeds,d,Foot injury - 50% chance of playing,1.50,0.112,0.50,0.056
603,Sávio,Spurs,i,Muscular injury - Expected back 21 Nov,0.54,0.043,1.00,0.043
523,Rashford,Man Utd,i,Hip injury - Unknown return date,0.78,0.043,1.00,0.043
337,Belloumi,Hull City,d,Hamstring injury - 75% chance of playing,1.29,0.135,0.25,0.034
202,Thomas,Coventry City,d,Calf injury - 50% chance of playing,0.60,0.063,0.50,0.031



Total share of team xG currently missing:


,xg_share_lost
team_name,
Brentford,0.315
Chelsea,0.270
Arsenal,0.228
Crystal Palace,0.169
Sunderland,0.102
Man Utd,0.095
Spurs,0.066
Leeds,0.058
Hull City,0.050


In [20]:
from src.evaluate import compare_totals
bt = backtest(matches, test_season=last_full)   # rerun so bt includes the new market columns
compare_totals(bt, train_matches=prior)

ImportError: cannot import name 'compare_totals' from 'src.evaluate' (/content/epl-score-predictor/src/evaluate.py)